In [ ]:
import numpy as np
import pandas as pd
from scipy.special import softmax
import os


EVAL_CSV = "../data/processed/evaluation_processed.csv"
OUT_DIR  = "../data/submission/ensamble_submission"

LOGITS = {
	"roberta_512": "roberta_processed/logits_roberta_processed_noseed_MAXLEN_512.npy",
	"roberta_256": "roberta_MAXLEN256/logits_roberta_processed_noseed_MAXLEN_256.npy",
	"deberta_512": "deberta_processed/logits_deberta_processed_noseed_MAXLEN_512.npy",
	"deberta_256": "deberta_MAXLEN256/logits_deberta_processed_noseed_MAXLEN_256.npy",
}

WEIGHTS = {
	"roberta_512": 0.30,
	"roberta_256": 0.25,
	"deberta_512": 0.30,
	"deberta_256": 0.15,
}

HARDCASE_PERCENTILE = 70   
N_CLASSES = 7

os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
#LOAD DATA
df_eval = pd.read_csv(EVAL_CSV)
ids = df_eval["Id"].values

logits = {}
for name, path in LOGITS.items():
	logits[name] = np.load(path)
	print(f"{name}: {logits[name].shape}")

N, C = next(iter(logits.values())).shape
assert C == N_CLASSES

In [ ]:
#UTILS
def save_submission(preds, name):
	pd.DataFrame({
		"Id": ids,
		"Predicted": preds.astype(int)
	}).to_csv(f"{OUT_DIR}/{name}.csv", index=False)

def entropy(p):
	return -np.sum(p * np.log(p + 1e-12), axis=1)


In [ ]:
#SIMPLE AVERAGE
avg_logits = np.mean(list(logits.values()), axis=0)
avg_preds  = avg_logits.argmax(axis=1)

save_submission(avg_preds, "submission_avg")


In [ ]:
#WEIGHTED AVERAGE
weighted_logits = np.zeros_like(avg_logits)
for name, w in WEIGHTS.items():
	weighted_logits += w * logits[name]

weighted_preds = weighted_logits.argmax(axis=1)
save_submission(weighted_preds, "submission_weighted")


In [ ]:
#Roberta Only
rob_logits = (
	0.5 * logits["roberta_512"] +
	0.5 * logits["roberta_256"]
)
rob_preds = rob_logits.argmax(axis=1)

save_submission(rob_preds, "submission_roberta_only")

In [ ]:
# DEberta only 

deb_logits = (
	0.5 * logits["deberta_512"] +
	0.5 * logits["deberta_256"]
)
deb_preds = deb_logits.argmax(axis=1)

save_submission(deb_preds, "submission_deberta_only")

In [ ]:
#HardCase Routing 
base_probs = softmax(weighted_logits, axis=1)
H = entropy(base_probs)
threshold = np.percentile(H, HARDCASE_PERCENTILE)

rob_probs = softmax(rob_logits, axis=1)

final_preds = np.where(
	H > threshold,
	rob_probs.argmax(axis=1),
	base_probs.argmax(axis=1)
)
save_submission(final_preds, "submission_hardcase")

In [ ]:
# WEIGHTED + HARD-CASE + ARCH ROUTING
deb_probs = softmax(deb_logits, axis=1)

rob_conf = rob_probs.max(axis=1)
deb_conf = deb_probs.max(axis=1)

hard_mask = H > threshold

final_preds = base_probs.argmax(axis=1)  # default = weighted ensemble

choose_rob = rob_conf > deb_conf

final_preds[hard_mask & choose_rob] = rob_probs.argmax(axis=1)[hard_mask & choose_rob]
final_preds[hard_mask & (~choose_rob)] = deb_probs.argmax(axis=1)[hard_mask & (~choose_rob)]

save_submission(final_preds, "submission_weighted_hardcase")